# 🪔 FinVaani — Training on Google Colab T4
**Indian Finance Q&A System using Lottery Ticket Hypothesis**

This notebook trains the full FinVaani pipeline on Colab T4 GPU, then saves the model to Google Drive for transfer back to your local machine.

**Estimated time:** ~60 min on T4 | ~20 min on A100
**Runtime:** GPU (Runtime → Change runtime type → T4 GPU)

## Step 1: Check GPU & Mount Drive

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

# Mount Google Drive (model will be saved here)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/finvaani_models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Models will be saved to: {DRIVE_DIR}")

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install transformers>=4.36.0 peft>=0.7.0 datasets>=2.14.0 evaluate>=0.4.0 accelerate>=0.24.0 rouge-score nltk tqdm pandas scikit-learn
print("All dependencies installed.")

## Step 3: Upload or Clone Project Data

In [ ]:
import os

# Option A: If you have the project on Drive already
# PROJECT_DIR = '/content/drive/MyDrive/finvaani'

# Option B: Upload data splits directly (recommended)
# Upload your data/splits/ folder to Colab using the file browser
# or run the scraping + preprocessing here

# For this notebook we'll create the data inline from the uploaded CSVs
# Make sure you've uploaded: train.csv, val.csv, test.csv to /content/

TRAIN_CSV = '/content/train.csv'
VAL_CSV   = '/content/val.csv'
TEST_CSV  = '/content/test.csv'

# Check files exist
for f in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if os.path.exists(f):
        import pandas as pd
        df = pd.read_csv(f)
        print(f"{f}: {len(df)} rows")
    else:
        print(f"MISSING: {f} — please upload this file")

## Step 4: LoRA Fine-tuning (Full Dataset, All 3 Epochs)

In [ ]:
import torch
import math
import time
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "ai-forever/mGPT"
OUTPUT_DIR = "/content/lora_finetuned"
MAX_LENGTH = 256
SEED = 42

torch.manual_seed(SEED)
device = "cuda"
print(f"Training on: {device.upper()}")
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Load tokenizer and model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading mGPT (1.4B params) in fp16...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.eos_token_id
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Apply LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Tokenize full dataset
def tokenize_dataset(csv_path, max_length=MAX_LENGTH):
    df = pd.read_csv(csv_path).dropna(subset=["formatted"])
    print(f"  Loaded {len(df)} rows from {csv_path}")
    
    def tokenize(batch):
        tokens = tokenizer(
            batch["formatted"],
            truncation=True, max_length=max_length, padding="max_length",
        )
        tokens["labels"] = tokens["input_ids"].copy()
        return tokens
    
    ds = Dataset.from_pandas(df[["formatted"]])
    ds = ds.map(tokenize, batched=True, remove_columns=["formatted"])
    ds.set_format("torch")
    return ds

print("Tokenizing datasets (full, no subset)...")
train_ds = tokenize_dataset(TRAIN_CSV)
val_ds   = tokenize_dataset(VAL_CSV)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# Estimate time
steps = math.ceil(len(train_ds) / (4 * 4)) * 3  # batch=4, grad_accum=4, 3 epochs
print(f"Total steps: {steps} (~{steps * 1.2 / 60:.0f} min on T4)")

In [ ]:
# Training arguments — optimised for T4 16GB
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    optim="adamw_torch_fused",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting training...")
t_start = time.time()
trainer.train()
elapsed = time.time() - t_start
print(f"\nTraining complete in {elapsed/60:.1f} minutes")

## Step 5: Save Model to Google Drive

In [ ]:
import shutil

# Save LoRA adapter (only ~20MB — NOT the full 1.4B model)
print("Saving LoRA adapter...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Copy to Google Drive
drive_lora_dir = f"{DRIVE_DIR}/lora_finetuned"
if os.path.exists(drive_lora_dir):
    shutil.rmtree(drive_lora_dir)
shutil.copytree(OUTPUT_DIR, drive_lora_dir)

# List saved files and sizes
print(f"\nSaved to Google Drive: {drive_lora_dir}")
for f in os.listdir(drive_lora_dir):
    size = os.path.getsize(os.path.join(drive_lora_dir, f))
    print(f"  {f}: {size/1e6:.1f} MB")

total_size = sum(
    os.path.getsize(os.path.join(drive_lora_dir, f))
    for f in os.listdir(drive_lora_dir)
)
print(f"\nTotal adapter size: {total_size/1e6:.1f} MB")
print("(Base model stays on HuggingFace — no need to transfer 5.6GB)")

## Step 6: Quick Validation

In [ ]:
# Test the fine-tuned model on sample questions
model.eval()

def generate(question, language="en"):
    if language == "hi":
        prompt = f"### सवाल: {question}\n### जवाब:"
    else:
        prompt = f"### Question: {question}\n### Answer:"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=100, do_sample=True,
            temperature=0.7, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    for stop in ["<|endoftext|>", "### Question:", "### सवाल:", "\n\n\n"]:
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

test_questions = [
    ("What is the repo rate and how does RBI use it?", "en"),
    ("What is a Non-Banking Financial Company (NBFC)?", "en"),
    ("What is UPI and how does it work?", "en"),
    ("रेपो रेट क्या है?", "hi"),
    ("म्यूचुअल फंड क्या है?", "hi"),
]

print("=" * 60)
for q, lang in test_questions:
    ans = generate(q, lang)
    tag = "[EN]" if lang == "en" else "[HI]"
    print(f"{tag} Q: {q}")
    print(f"     A: {ans[:150]}")
    print()

## Step 7: LTH Pruning (Iterative Magnitude Pruning)

In [ ]:
import copy
import torch.nn.utils.prune as prune

# Load original weights for LTH reset
original_state_dict = copy.deepcopy(model.state_dict())
print("Original weights saved for LTH reset.")

def get_lora_modules(model):
    return [(n, m) for n, m in model.named_modules()
            if hasattr(m, "lora_A") and hasattr(m, "lora_B")]

def get_sparsity(model):
    total = zeros = 0
    for _, module in get_lora_modules(model):
        for key in ["default"]:
            for d in [module.lora_A, module.lora_B]:
                if key in d:
                    w = d[key].weight
                    total += w.numel()
                    zeros += (w == 0).sum().item()
    return 100.0 * zeros / max(total, 1)

def prune_lora(model, amount=0.20):
    for _, module in get_lora_modules(model):
        for key in ["default"]:
            for d in [module.lora_A, module.lora_B]:
                if key in d:
                    prune.l1_unstructured(d[key], name="weight", amount=amount)
    return model

def reset_weights_lth(model, original_state_dict):
    """KEY LTH STEP: reset values to original, keep mask."""
    with torch.no_grad():
        for name, module in get_lora_modules(model):
            for key in ["default"]:
                for lk, ld in [("lora_A", module.lora_A), ("lora_B", module.lora_B)]:
                    if key in ld:
                        linear = ld[key]
                        orig_key = f"base_model.model.{name}.{lk}.{key}.weight"
                        if orig_key in original_state_dict:
                            mask = linear.weight_mask if hasattr(linear, "weight_mask") \
                                   else (linear.weight != 0).float()
                            linear.weight.data.copy_(
                                original_state_dict[orig_key].to(linear.weight.device))
                            linear.weight.data.mul_(mask)
    return model

print("LTH utility functions ready.")

In [ ]:
# IMP Loop — 5 rounds, 20% pruning per round
PRUNE_AMOUNT  = 0.20
MAX_ROUNDS    = 5
QUALITY_FLOOR = 0.85

# Baseline eval loss
def get_eval_loss(model):
    from torch.utils.data import DataLoader
    loader = DataLoader(val_ds, batch_size=4)
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            lbl = batch["labels"].to(device)
            out = model(input_ids=ids, labels=lbl)
            total_loss += out.loss.item()
            n += 1
            if n >= 30: break
    return total_loss / max(n, 1)

baseline_loss = get_eval_loss(model)
print(f"Baseline eval loss: {baseline_loss:.4f}")

best_round = 0
best_loss  = baseline_loss

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n{'='*50}")
    print(f"PRUNING ROUND {round_num}/{MAX_ROUNDS}")
    
    # a) Prune
    model = prune_lora(model, PRUNE_AMOUNT)
    sparsity = get_sparsity(model)
    print(f"Sparsity: {sparsity:.1f}%")
    
    # b) LTH reset
    model = reset_weights_lth(model, original_state_dict)
    
    # c) Retrain 1 epoch
    retrain_args = TrainingArguments(
        output_dir=f"/content/pruned_round_{round_num}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        learning_rate=2e-4, fp16=True,
        logging_steps=50, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none", seed=SEED,
        dataloader_num_workers=2, dataloader_pin_memory=True,
    )
    trainer_r = Trainer(
        model=model, args=retrain_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    trainer_r.train()
    
    # d) Evaluate
    round_loss = get_eval_loss(model)
    loss_ratio = round_loss / max(baseline_loss, 1e-6)
    print(f"Round {round_num} loss: {round_loss:.4f} (ratio: {loss_ratio:.2f})")
    
    # e) Save round checkpoint to Drive
    round_dir = f"{DRIVE_DIR}/pruned_round_{round_num}"
    model.save_pretrained(round_dir)
    tokenizer.save_pretrained(round_dir)
    print(f"Saved round {round_num} → {round_dir}")
    
    # f) Check quality floor (using loss ratio as proxy)
    if loss_ratio <= (1.0 / QUALITY_FLOOR):
        best_round = round_num
        best_loss  = round_loss
        print(f"Quality maintained. Continuing.")
    else:
        print(f"Quality degraded. Round {round_num-1} is the winning ticket.")
        break

print(f"\nWinning ticket: Round {best_round} (sparsity ~{best_round*16:.0f}%)")

In [ ]:
# Save winning ticket to Drive
import shutil

winning_src = f"{DRIVE_DIR}/pruned_round_{best_round}"
winning_dst = f"{DRIVE_DIR}/winning_ticket"

if os.path.exists(winning_dst):
    shutil.rmtree(winning_dst)
if os.path.exists(winning_src):
    shutil.copytree(winning_src, winning_dst)
    print(f"Winning ticket saved → {winning_dst}")
else:
    # Fallback: save current model state
    model.save_pretrained(winning_dst)
    tokenizer.save_pretrained(winning_dst)
    print(f"Winning ticket (current state) saved → {winning_dst}")

# Final summary
print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Files saved to Google Drive: {DRIVE_DIR}/")
for item in os.listdir(DRIVE_DIR):
    item_path = os.path.join(DRIVE_DIR, item)
    if os.path.isdir(item_path):
        size = sum(os.path.getsize(os.path.join(item_path, f))
                   for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f)))
        print(f"  {item}/  ({size/1e6:.1f} MB)")

## Step 8: Download to Your Mac

In [ ]:
# Option A: Download directly from Colab to your Mac
# Run this to zip and download the winning ticket adapter

import zipfile

zip_path = "/content/finvaani_winning_ticket.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(winning_dst):
        zf.write(os.path.join(winning_dst, fname), fname)

from google.colab import files
files.download(zip_path)
print(f"Downloading {zip_path}...")
print("After download, on your Mac:")
print("  unzip finvaani_winning_ticket.zip -d finvaani/models/winning_ticket/")

In [ ]:
# Option B: The files are already on Google Drive
# On your Mac, use Google Drive desktop app or:
# 1. Go to drive.google.com
# 2. Navigate to MyDrive/finvaani_models/winning_ticket/
# 3. Download the folder
# 4. Place contents in: finvaani/models/winning_ticket/
# 5. Also download lora_finetuned/ → finvaani/models/lora_finetuned/

print("Files on Google Drive:")
print(f"  {DRIVE_DIR}/lora_finetuned/    ← LoRA fine-tuned adapter")
print(f"  {DRIVE_DIR}/winning_ticket/    ← Best pruned adapter")
print()
print("On your Mac, place them at:")
print("  finvaani/models/lora_finetuned/")
print("  finvaani/models/winning_ticket/")
print()
print("The base mGPT model is already cached at:")
print("  ~/.cache/huggingface/hub/models--ai-forever--mGPT/")
print("No need to re-download the 5.6GB base model!")

## GPU Memory Monitor

In [ ]:
# Run this any time to check GPU memory
def gpu_stats():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {allocated:.2f}GB allocated | {reserved:.2f}GB reserved | {total:.1f}GB total")
    print(f"Free: {total - reserved:.2f}GB")

gpu_stats()